In [26]:
# import libraries
from openai import OpenAI
from openai import AsyncOpenAI
import asyncio
import time
import math
import random
from dotenv import load_dotenv
import os
import sys
from pathlib import Path
import json
import re
import sys
from seqeval.metrics import classification_report as seqeval_classification_report

# set path to project root and import custom classes and functions
base_path = Path.cwd() / "../../../"
sys.path.append(str(base_path.resolve()))
from utils.evaluation import extract_spans, mention_level_evaluation

In [2]:
def create_llm_annotations(text, spans):

    # sort the spans according to their start and end index
    spans = sorted(spans, key=lambda x: x["start"])

    # store the text and set index variable
    llm_text = ""
    last_idx = 0

    # loop through spans and add the span with custom characters
    for span in spans:
        llm_text += text[last_idx:span["start"]]
        llm_text += f"@@{text[span["start"]:span["end"]]}##"
        last_idx = span["end"]

    # add the rest of the text
    llm_text += text[last_idx:]

    return llm_text

def llm_output_to_bio(annotated_text):

    # split words via a regex
    words = re.findall(r"@@.*?##|\w+|'\w+|[^\w\s]", annotated_text)

    # empty list to store the bio tags
    bio_tags = []

    # loop through all words
    for word in words:

        # if it is an annotated span, split words and assign bio labels
        if word.startswith("@@") and word.endswith("##"):
            entity_text = word[2:-2]
            entity_words = re.findall(r"\w+|'\w+|[^\w\s]", entity_text)
            for i, t in enumerate(entity_words):
                tag = "B-sg" if i == 0 else "I-sg"
                bio_tags.append((t, tag))
        else:
            # otherwise assign O tag
            bio_tags.append((word, "O"))

    return bio_tags

In [3]:
# initialize empty dataset list
training_data = []
validation_data = []

with open("../../../01_data/training_validation_set/training_set.json", "r") as f:
    raw_training_data = json.load(f)

with open("../../../01_data/training_validation_set/validation_set.json", "r") as f:
    raw_val_data = json.load(f)

# loop through all sentences in the data
for task in raw_training_data:
    # get the sentence and all annotations
    text = task["sentence"]
    spans = task["annotations"]
    labels = [annotation["text"] for annotation in spans]
   
    llm_text = create_llm_annotations(text, spans)
    bio_tags = llm_output_to_bio(llm_text)

    # add everything to the dataset list
    training_data.append({
        "text": text,
        "labels": labels,
        "llm_text": llm_text,
        "bio_tags": bio_tags
    })

# loop through all sentences in the data
for task in raw_val_data:
    # get the sentence and all annotations
    text = task["sentence"]
    spans = task["annotations"]
    labels = [annotation["text"] for annotation in spans]
   
    llm_text = create_llm_annotations(text, spans)
    bio_tags = llm_output_to_bio(llm_text)

    # add everything to the dataset list
    validation_data.append({
        "text": text,
        "labels": labels,
        "llm_text": llm_text,
        "bio_tags": bio_tags
    })

In [107]:
# do an estimation of the costs for running inference with different models
model_costs = {
    "4o-mini":
    {"standard": {
        "input": 0.15,
        "output": 0.6
    },
    "batch": {
        "input": 0.075,
        "output": 0.3
    }},
    "4o":
     {"standard": {
        "input": 2.5,
        "output": 10
    },
    "batch": {
        "input": 1.25,
        "output": 5
    }},
    "5-nano": 
    {"standard": {
        "input": 0.05,
        "output": 0.4
    },
    "batch": {
        "input": 0.025,
        "output": 0.2
    }
}}

input_lengths = [1500, 2000, 3000]
token_multiple = 1.3
test_type = "Validation"
gpt_mode = "standard"

print(f"{test_type} Phase with {gpt_mode} processing")
print("-"*70)
for model in model_costs:
    for input in input_lengths:
        if test_type == "Validation":
            input_costs = (input*token_multiple*len(validation_data)/1000000)*model_costs[model][gpt_mode]["input"]
            output_costs = (75*token_multiple*len(validation_data)/1000000)*model_costs[model][gpt_mode]["output"]
        elif test_type == "CV":
            input_costs = (input*token_multiple*5000/1000000)*model_costs[model][gpt_mode]["input"]
            output_costs = (75*token_multiple*5000/1000000)*model_costs[model][gpt_mode]["output"]
        elif test_type == "Inference":
            input_costs = (input*token_multiple*500000/1000000)*model_costs[model][gpt_mode]["input"]
            output_costs = (75*token_multiple*500000/1000000)*model_costs[model][gpt_mode]["output"]
        total_costs = input_costs + output_costs
        print(f"Total costs for {model} with input prompt of length {input}: {total_costs:.4f}")
    print("-"*70)

Validation Phase with standard processing
----------------------------------------------------------------------
Total costs for 4o-mini with input prompt of length 1500: 0.3510
Total costs for 4o-mini with input prompt of length 2000: 0.4485
Total costs for 4o-mini with input prompt of length 3000: 0.6435
----------------------------------------------------------------------
Total costs for 4o with input prompt of length 1500: 5.8500
Total costs for 4o with input prompt of length 2000: 7.4750
Total costs for 4o with input prompt of length 3000: 10.7250
----------------------------------------------------------------------
Total costs for 5-nano with input prompt of length 1500: 0.1365
Total costs for 5-nano with input prompt of length 2000: 0.1690
Total costs for 5-nano with input prompt of length 3000: 0.2340
----------------------------------------------------------------------


In [27]:
# retrieve some few-shot examples
# create some few-shot examples
positive_examples = [ex for ex in validation_data if ex["labels"]]
negative_examples = [ex for ex in validation_data if not ex["labels"]]
# few_shot_examples = random.sample(non_empty_examples, 4) + random.sample(empty_examples, 1)

In [203]:
manual_detailed_description = """
## Task Objective
You label mentions of social groups within single sentences. Mark all spans according to the instructions that meet the definition of a social group.  
If you are uncertain whether a mention qualifies, do not label it.
                        
## Definition of a Social Group
A social group is a collection of people who share **socio-demographic attributes** (e.g. "working people").
Institutions (e.g. "business, "Government") **are not** social groups.
However, **groups of individuals** within institutions **are** social groups if they have a shared socio-demographic attribute (e.g. "business people").
Groups defined by political opinion as well as the individuals within them (e.g. "Labour party", "Labour politicians") **are not** social groups.
Individual persons or highly specifc collectives (e.g. "family of my hon. friend") are **not** social groups.
General terms (e.g. "communities", "people") count **only** particular group is specified ("local communities", "our people").

## Positive Examples to Label
This is a list of positive examples that **should** be labeled as social group: "Families", "constituents", "employers", "victims"

## Negative Examples Not to Label
This is a list of negative examples that **should not** be labeled as social group: "people", "business", "EU", "taxpayers"

## Specific Instructions
- Label only the social group component and not articles, modifying terms ("any", "every") or numerical descriptors
- Only label singular forms if it is a generalization to a broader group of people (e.g. "every young person") or it is part of a composite term ("child poverty"). Label only the social group component ("young person", "child").
- If a social group is mentioned indirectly as part of a name (e.g. "Society for Prevention of Child Cruelty"), label only the social group component ("Child").
- Do **not** label verbal forms (e.g. "farming")
- If a group is mentioned using a genitive form ("women's lives"), label only the social group component without the genitive ending ("women").
- If groups are linked with conjunctions but each has a distinct social-group identity (e.g. "women and children"), label them separately unless the meaning depends on the combined phrase as a single unit (e.g. "veterans, their wives and their children").

## Output Format
Return the full sentence without paraphrasing. Mark the start of each social group mention with @@ and the end with ##.
If no social groups are present, return the sentence unchanged.
"""

manual_medium_description = """
## Task Objective
Label mentions of social groups in single sentences. Mark all qualifying spans. If uncertain whether a mention qualifies, do not label it.

## Definition of a Social Group
A social group is a set of people sharing **socio-demographic attributes** ("working people").  
**Do not** label:
- institutional groups, state authorities, or countries ("business", "Government")
- groups defined by political opinion or beliefs ("Labour party", "Conservatives")
- individual persons or highly specific collectives ("family of my hon. colleague")
**Do** label:
- subgroups of institutional entities with shared socio-demographic traits ("business people")  
General terms ("communities", "people") count **only** particular group is specified ("local communities", "our people").

## Positive Examples
"Families", "constituents", "employers", "victims"

## Negative Examples
"people", "business", "EU", "taxpayers"

## Specific Instructions
- Label only the social group component and do **not** label articles, modifying terms ("any", "every", "per") or numerical descriptors
- Label singular forms only if generalizing ("every young person") or part of a composite ("child poverty")
- Label additional descriptions of the social group if it is part of their sociodemographic attribute ("working people in Leicester")
- Do **not** label verbal forms ("farming")
- For policy/program titles ("Society for Prevention of Child Cruelty"), label only the embedded group ("Child")
- For genitive forms ("women's lives"), label **only** the social group component ("women")

## Output Format
Return the full sentence without any paraphrasing. Mark each social group with:
- "@@" at the start  
- "##" at the end  
If no social groups are present, return the sentence unchanged.
"""

manual_short_description = """
## Task Objective
Label mentions of social groups in single sentences. If uncertain whether a mention qualifies, do not label it. Never paraphrase the sentence in any way!

## Definition of a Social Group
A social group is a collective of people sharing **socio-demographic attributes** ("working people").  
**Do not** label:
- institutional groups ("business", "Government")  
- groups defined by political opinion or beliefs ("Labour party", "Conservatives")
- individual persons or highly specific collectives ("family of a colleague")
**Do** label:
- subgroups of institutions with shared socio-demographic traits ("business people")  
General terms ("communities") count **only** particular group is specified ("local communities").

## Specific Instructions
- Label only the social group component and do **not** label articles, modifying terms ("any", "every", "per") or numerical descriptors
- Label additional descriptions of the social group if it is part of their sociodemographic attribute ("working people in Leicester")
- Only label singular mentions if it is a generlization to a broader group
- Do not label verbal forms

## Output Format
Return the full sentence without any paraphrasing. Mark each social group with:
- "@@" at the start  
- "##" at the end  
If no social groups are present, return the sentence unchanged.
"""

non_paraphrase_reminder = """
You have generated a paraphrase. Stick closely to the rules without paraphrasing the sentence in any way!
"""

manual_undercomplex_description = """
## Task Objective
Label mentions of social groups in single sentences. If uncertain whether a mention qualifies, do not label it.

## Definition of a Social Group
A social group is a collective of people sharing **socio-demographic attributes**. **Do not** label institutional groups, but **do label** groupings of people within an institution.

## Positive Examples
"Families", "constituents", "employers", "victims"

## Negative Examples
"people", "business", "EU", "taxpayers"

## Output Format
Return the full sentence without any paraphrasing. Mark each social group with:
- "@@" at the start  
- "##" at the end  
If no social groups are present, return the sentence unchanged.
"""

In [140]:
def compile_prompt_summarization(prompt_to_summarize):
    chat = [
        {
            "role": "system",
            "content": """
            Rewrite the system prompt below into a shorter, clearer system prompt suitable for directly instructing a GPT model in a Named Entity Recognition task.
            The rewritten version must:
            - preserve all definitions and classification criteria exactly,
            - preserve all span-selection rules,
            - maintain the output-format specification,
            - remove repetition and unnecessary explanation,
            - express rules in a direct, imperative style,
            - remain self-contained.
            Output only the rewritten prompt."""
        },
        {
            "role": "user",
            "content": prompt_to_summarize
        }
    ]
    return chat

prompt = compile_prompt_summarization(manual_detailed_description)
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=prompt
)
synthetically_distilled_prompt = response.choices[0].message.content.strip()

In [216]:
# compile manual few-shot examples
positive_examples = [
    {"text": "Our party encourages all @@employers in Leicester## to hire those with physical disabilities.",
     "llm_text": "Our party encourages all @@employers in Leicester## to hire @@those with physical disabilities##."},
     {"text": "I would like to make the voice of my constituents heard on the issue of victims and veterans support.",
     "llm_text": "I would like to make the voice of my @@constituents## heard on the issue of @@victims## and @@veterans## support."}
]

negative_examples = [
    {"text": "The Government and the Labour party are working together on this matter.",
     "llm_text": "The Government and the Labour party are working together on this matter."},
      {"text": "Small businesses in this country are vital for our economy.",
     "llm_text": "Small businesses in this country are vital for our economy."}
]

few_shot_examples = """
Sentence: Our party encourages all @@employers in Leicester## to hire those with physical disabilities.
Our party encourages all @@employers in Leicester## to hire @@those with physical disabilities##.

Sentence: The Government and the Labour party are working together on this matter.
The Government and the Labour party are working together on this matter.

Sentence: I would like to make the voice of my constituents heard on the issue of victims and veterans support.
I would like to make the voice of my @@constituents## heard on the issue of @@victims## and @@veterans## support.

Sentence: Small businesses in this country are vital for our economy.
Small businesses in this country are vital for our economy.

"""

In [257]:
# create different prompt templates for testing performance on the validation set
def compile_prompt_ner(system_prompt, non_paraphrase_reminder, positive_examples, negative_examples, test_sentence, paraphrase_reminder = False):

        if paraphrase_reminder == False:
                chat = [
                        {
                                "role": "system",
                                "content": system_prompt
                        }]
        elif paraphrase_reminder == True:
                chat = [
                        {
                             "role": "system",
                             "content": non_paraphrase_reminder + system_prompt 
                        }
                ]

        # add a positive example
        chat.append({"role": "user", "content": f"Sentence: {positive_examples[0]['text']}"})
        chat.append({"role": "assistant", "content": positive_examples[0]["llm_text"]})

        # add a negative example
        chat.append({"role": "user", "content": f"Sentence: {negative_examples[0]['text']}"})
        chat.append({"role": "assistant", "content": negative_examples[1]["llm_text"]})

        # add a positive example
        chat.append({"role": "user", "content": f"Sentence: {positive_examples[1]['text']}"})
        chat.append({"role": "assistant", "content": positive_examples[1]["llm_text"]})

        # add a negative example
        chat.append({"role": "user", "content": f"Sentence: {negative_examples[1]['text']}"})
        chat.append({"role": "assistant", "content": negative_examples[1]["llm_text"]})
        
        # add the test sentence
        chat.append({"role": "user", "content": f"Sentence: {test_sentence}"})

        return chat

def compile_few_shot_examples(positive_examples, negative_examples, test_sentence):
        
        chat = []
        
        # add a positive example
        chat.append({"role": "user", "content": f"Sentence: {positive_examples[0]['text']}"})
        chat.append({"role": "assistant", "content": positive_examples[0]["llm_text"]})

        # add a negative example
        chat.append({"role": "user", "content": f"Sentence: {negative_examples[0]['text']}"})
        chat.append({"role": "assistant", "content": negative_examples[1]["llm_text"]})

        # add a positive example
        chat.append({"role": "user", "content": f"Sentence: {positive_examples[1]['text']}"})
        chat.append({"role": "assistant", "content": positive_examples[1]["llm_text"]})

        # add a negative example
        chat.append({"role": "user", "content": f"Sentence: {negative_examples[1]['text']}"})
        chat.append({"role": "assistant", "content": negative_examples[1]["llm_text"]})
        
        # add the test sentence
        chat.append({"role": "user", "content": f"Sentence: {test_sentence}"})

        return chat

In [310]:
# create dictionary storing rate limits
model_limits = {
    "gpt-4o-mini":
    {"token_limit": 200000,
     "request_limit": 500
    },
    "gpt-4o":
    {"token_limit": 30000,
     "request_limit": 500
    },
    "gpt-5-nano":
    {"token_limit": 200000,
     "request_limit": 500
    }
    }

# select the model to use
model = "gpt-5-nano"

# select which system message to use
system_message = manual_medium_description

# create client for interacting with API
load_dotenv()
client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# select approximate token frequencies
input_freq = 5000
output_freq = 100
reasoning = 300
token_multiple = 1.5
if model == "gpt-4o-mini":
    total_length = (input_freq+output_freq+reasoning)*token_multiple
else:
    total_length = (input_freq+output_freq)*token_multiple

# calculate a safe request rate
max_req_tpm = math.floor(model_limits[model]["token_limit"]/total_length)
safe_rpm = min(max_req_tpm, model_limits[model]["request_limit"])


safe_interval = 60.0 / safe_rpm
max_concurrency = 100

sem = asyncio.Semaphore(max_concurrency)
last_call = 0

async def send_limited(prompt, sentence, max_retries=2):
    global last_call
    attempt = 0
    original_words = sentence.strip().split()

    while attempt <= max_retries:
        async with sem:
            # rate limiting
            now = time.time()
            wait = last_call + safe_interval - now
            if wait > 0:
                await asyncio.sleep(wait)
            last_call = time.time()

            # send the request with responses api
            response = await client.chat.completions.create(
                model = model,
                messages = prompt,
                reasoning_effort="medium"
            )
            output_text = response.choices[0].message.content

        # check if output matches original in word count
        if len(output_text.strip().split()) == len(original_words):
            return output_text

        # retry
        attempt += 1
        prompt = compile_prompt_ner(system_message, non_paraphrase_reminder, positive_examples, negative_examples, sentence, paraphrase_reminder = True)

    # fallback if all retries fail
    return sentence  # return original sentence


async def run_all(validation_data):
    tasks = []
    for row in validation_data:
        sentence = row["text"]
        prompt = compile_prompt_ner(system_message, non_paraphrase_reminder, positive_examples, negative_examples, sentence, paraphrase_reminder = False)
        tasks.append(asyncio.create_task(send_limited(prompt, sentence)))
    return await asyncio.gather(*tasks)

In [311]:
random_indices = random.sample(range(1000), 100)
val_subset = [validation_data[i] for i in random_indices]
output_texts = await run_all(val_subset)
output_bios = [llm_output_to_bio(text) for text in output_texts]

In [312]:
# conduct manual error analysis
for idx in range(0, 100):
    print(val_subset[idx]["llm_text"])
    print(output_texts[idx])
    print("-"*100)

We know the validity in that statement because 1,000 more people have been getting into work each and every day since 2010.
We know the validity in that statement because 1,000 more people have been getting into work each and every day since 2010.
----------------------------------------------------------------------------------------------------
We have seen 2 million new apprenticeship starts under this Government-a far, far higher rate of apprenticeship starts than ever occurred under 13 years of the Labour Government.
We have seen 2 million new apprenticeship starts under this Government-a far, far higher rate of apprenticeship starts than ever occurred under 13 years of the Labour Government.
----------------------------------------------------------------------------------------------------
I hope to see that followed through swiftly by the Treasury, which is working closely with Executive Ministers so that this issue can be devolved as soon as possible.
I hope to see that follow

In [313]:
# evaluate the generated answers

# get list of bio tags only
ground_truth_bio = [[tag for (_, tag) in sent["bio_tags"]] for sent in val_subset]
pred_bio = [[tag for (_, tag) in sent] for sent in output_bios]

filtered_gt = []
filtered_pred = []
no_match = []
idx = 0
for gt, pred in zip(ground_truth_bio, pred_bio):
    idx += 1
    if len(gt) == len(pred):
        filtered_gt.append(gt)
        filtered_pred.append(pred)
    else:
        no_match.append(idx)

y_true = [tag for sent in filtered_gt for tag in sent]
y_pred = [tag for sent in filtered_pred for tag in sent]

# evaluate at the entity level with seqeval
print(f"Number of non-matching sentences: {len(no_match)}\n")
print(seqeval_classification_report(filtered_gt, filtered_pred))

# compute cross-span evaluation
all_true_spans = []
all_predicted_spans = []

for idx in range(len(filtered_gt)):

    # get the spans
    all_true_spans.append(extract_spans(filtered_gt[idx]))
    all_predicted_spans.append(extract_spans(filtered_pred[idx]))

# apply cross-span evaluation
mention_level_evaluation(all_true_spans, all_predicted_spans)

Number of non-matching sentences: 1

              precision    recall  f1-score   support

          sg       0.72      0.64      0.68        69

   micro avg       0.72      0.64      0.68        69
   macro avg       0.72      0.64      0.68        69
weighted avg       0.72      0.64      0.68        69



{'precision': 0.6133477633477633,
 'recall': 0.6818181818181818,
 'f1': 0.6234384662956092}

In [241]:
def extract_final_answer(text):
    # If output contains "Answer:", extract only what comes after.
    match = re.search(r'Answer:\s*(.*)', text, flags=re.DOTALL)
    if match:
        return match.group(1).strip()
    return text.strip()

validation_data_test = validation_data[50:250]

gen_answers = []

for item in validation_data_test:
    sentence = item["text"]

    prompt = compile_prompt_cot(cot_hidden_reasoning, sentence)

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=prompt,
        temperature=0
    )

    raw_output = response.choices[0].message.content

    # NEW: clean up and extract the literal final answer
    cleaned_output = extract_final_answer(raw_output)

    # Convert to BIO tags
    output_bio = llm_output_to_bio(cleaned_output)

    gen_answers.append({
        "text": cleaned_output,
        "bio": output_bio
    })


KeyboardInterrupt: 

In [244]:
for i in range(0, 70):
    print(f"Sentence {i+1}")
    print(validation_data_test[i]["llm_text"])
    print(gen_answers[i]["text"])
    print("-"*100)

Sentence 1
Last night we had a briefing from @@senior retired police officers## about the threat to national security from evidence that is being given in inquests in Northern Ireland that opens up the whole modus operandi of our security forces and security services.
Last night we had a briefing from @@senior retired police officers## about the threat to national security from evidence that is being given in inquests in Northern Ireland that opens up the whole modus operandi of our security forces and security services.
----------------------------------------------------------------------------------------------------
Sentence 2
We have set up three task and finish groups, and the third is looking specifically at the anomaly left over from regulations in the Care Standards Act 2000, whereby the police are unable to access information about @@children## in @@children##'s homes who go missing or get into trouble, in order to co-ordinate the action that needs to be taken to prevent thos

In [245]:
# evaluate the generated answers

# get list of bio tags only
ground_truth_bio = [[tag for (_, tag) in sent["bio_tags"]] for sent in validation_data_test]
pred_bio = [[tag for (_, tag) in sent["bio"]] for sent in gen_answers]

filtered_gt = []
filtered_pred = []
no_match = []
idx = 0
for gt, pred in zip(ground_truth_bio, pred_bio):
    idx += 1
    if len(gt) == len(pred):
        filtered_gt.append(gt)
        filtered_pred.append(pred)
    else:
        no_match.append(idx)

y_true = [tag for sent in filtered_gt for tag in sent]
y_pred = [tag for sent in filtered_pred for tag in sent]

# evaluate at the entity level with seqeval
print(seqeval_classification_report(filtered_gt, filtered_pred))

              precision    recall  f1-score   support

          sg       0.61      0.38      0.47       127

   micro avg       0.61      0.38      0.47       127
   macro avg       0.61      0.38      0.47       127
weighted avg       0.61      0.38      0.47       127



In [246]:
no_match

[79, 103, 118, 180]

In [247]:
all_true_spans = []
all_predicted_spans = []

for idx in range(len(filtered_gt)):

    # get the spans
    all_true_spans.append(extract_spans(filtered_gt[idx]))
    all_predicted_spans.append(extract_spans(filtered_pred[idx]))

# apply cross-span evaluation
mention_level_evaluation(all_true_spans, all_predicted_spans)

{'precision': 0.4611111111111111,
 'recall': 0.41395145437698627,
 'f1': 0.4122864920737261}